In [2]:
import numpy as np
from scipy.optimize import differential_evolution
import torch
from models import SHREDForecaster

In [15]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

def sig_prime(x):
    s = sigmoid(x)
    return s - s*s

def sig_pprime(x):
    s = sigmoid(x)
    return s - 3*s*s + 2*s*s*s

def tanh_prime(x):
    return 1//(np.cosh(x)*np.cosh(x))
def tanh_pprime(x):
    return -2*np.tanh(x)/(np.cosh(x)*np.cosh(x))




sig_prime_x1 = 0.5
sig_pprime_x1 = np.log(2 - np.sqrt(3))
sig_pprime_x2 = np.log(2 + np.sqrt(3))

tanh_prime_x1 = 0
tanh_pprime_x1 = -0.5*np.arccosh(2)
tanh_pprime_x2 = 0.5*np.arccosh(2)

sig_prime(0.5), sig_pprime(sig_pprime_x1), sig_pprime(sig_pprime_x2), tanh_pprime(tanh_pprime_x1), tanh_pprime(tanh_pprime_x2)

#p in [-2,2]
#p' in [-0.635,1.235]
#p'' in [-0.5,0.7]


(np.float64(0.2350037122015945),
 np.float64(0.09622504486493762),
 np.float64(-0.09622504486493733),
 np.float64(0.769800358919501),
 np.float64(-0.769800358919501))

In [16]:
f_primative = np.linspace(-2,2,100)
o_primative = np.linspace(-2,2,100)
i_primative = np.linspace(-2,2,100)
c__primative = np.linspace(-2,2,100)
c = np.linspace(-1,1,100)

o_f = lambda x: sigmoid(x)
o_prime_f = lambda x: sig_prime(x)
o_pprime_f = lambda x: sig_pprime(x)

p_f = lambda f,i,c,c_: sigmoid(f)*c * sigmoid(i)*np.tanh(c_)
p_prime_f = lambda f,i,c,c_: sig_prime(f)*c + sig_prime(i)*np.tanh(c_) + sigmoid(i)*tanh_prime(c_)
p_pprime_f = lambda f,i,c,c_: sig_pprime(f)*c + sig_pprime(i)*np.tanh(c_) + 2*sig_prime(i)*tanh_prime(c_) + sigmoid(i)*tanh_pprime(c_)

def activation_prime(f,i,o,c,c_):
    p = p_f(f,i,c,c_)
    p_prime = p_prime_f(f,i,c,c_)

    o_primative = o
    o = o_f(o_primative)
    o_prime = o_prime_f(o_primative)
    return o_prime*np.tanh(p) + o*p_prime*tanh_prime(p)

def activation_pprime(f,i,o,c,c_):
    p = p_f(f,i,c,c_)
    p_prime = p_prime_f(f,i,c,c_)
    p_pprime = p_pprime_f(f,i,c,c_)

    o_primative = o
    o = o_f(o_primative)
    o_prime = o_prime_f(o_primative)
    o_pprime = o_pprime_f(o_primative)
    return o_pprime*np.tanh(p) + 2*o_prime*p_prime*tanh_prime(p) + o*tanh_pprime(p)*p_prime*p_prime + o*tanh_prime(p)*p_pprime

Bound for the activation layers

In [17]:
# --- bounds for each parameter, matching your linspace ranges ---
bounds = [
    (-200, 200),  # f_primative
    (-200, 200),  # i_primative
    (-200, 200),  # o_primative
    (-1, 1),  # c
    (-200, 200),  # c__primatvive
]
 
def find_bounds(func, bounds, name=''):
    def neg_f(x):
        return -func(*x)
    
    def pos_f(x):
        return func(*x)
    
    res_max = differential_evolution(neg_f, bounds, tol=1e-12, seed=1, maxiter=2000, polish=True)
    res_min = differential_evolution(pos_f, bounds, tol=1e-12, seed=2, maxiter=2000, polish=True)
    print('Bounds for the function ', name)
    print("Max value:", -res_max.fun, "at f,i,o,c,c_ =", res_max.x)
    print("Min value:", res_min.fun, "at f,i,o,c,c_ =", res_min.x)

find_bounds(activation_prime, bounds, 'Activation prime')
find_bounds(activation_pprime, bounds, 'Activation pprime')
# cross-check with dense random sampling
# rng = np.random.default_rng(0)
# N = 2_000_000
# f = np.random.uniform(-2, 2, N)
# i = np.random.uniform(-2, 2, N)
# o = np.random.uniform(-2, 2, N)
# c = np.random.uniform(-1, 1, N)
# c_ = np.random.uniform(-2, 2, N)
# vals = activation_pprime(f, i, o, c, c_)
# print("Random sampling max:", vals.max())
# print("Random sampling min:", vals.min())


Bounds for the function  Activation prime
Max value: 0.2500000365002334 at f,i,o,c,c_ = [-9.81375869e-08 -1.71259470e+01  1.92763564e+02  1.00000000e+00
  1.12248851e+02]
Min value: -0.25000003650023744 at f,i,o,c,c_ = [-1.61840007e+01 -1.30888522e-07  1.39310010e+02 -3.89868334e-01
 -1.10835793e+02]
Bounds for the function  Activation pprime
Max value: 0.769800390529615 at f,i,o,c,c_ = [-15.98430901 144.3875373   58.61761156   0.27651811  -0.65847887]
Min value: -0.7698003905296256 at f,i,o,c,c_ = [-17.12530884  44.30572836  91.04434117  -0.86547276   0.65847896]


In [18]:
folder = 'forecaster_configs/'
j=3
forecaster_state_dict = torch.load(folder+f'config{j}_synth.json')
forecaster = SHREDForecaster(68, 64)
forecaster.load_state_dict(forecaster_state_dict)

<All keys matched successfully>

In [19]:
W = forecaster.lstm.lstm.weight_ih_l0.detach()
U = forecaster.lstm.lstm.weight_hh_l0.detach()
# b = forecaster.lstm.bias_ih_l0.detach()
W2 = forecaster.proj.weight.detach()
b2 = forecaster.proj.bias.detach()
np.linalg.norm(W, ord=2), np.linalg.norm(U, ord=2)
# , np.linalg.norm(b, ord=2), np.linalg.norm(W2, ord=2), np.linalg.norm(b2, ord=2)

(np.float32(7.886504), np.float32(5.2872496))

In [20]:
W_norm = np.linalg.norm(W, ord=2)
activation_prime_norm = 0.25
dzldx = [W_norm]
for j in range(1,11): #first layer to second last is to the end of the LSTM layers, coz last layer is proj
    dzldx.append(W_norm*activation_prime_norm*dzldx[-1])
print([_**2 for _ in dzldx])

[np.float32(62.19695), np.float32(241.77878), np.float32(939.86896), np.float32(3653.561), np.float32(14202.521), np.float32(55209.59), np.float32(214616.73), np.float32(834281.6), np.float32(3.2431108e+06), np.float32(1.2606974e+07), np.float32(4.9007204e+07)]


In [21]:
maxS = [W2.abs().max()]
L = seq_len = 10
for l in np.arange(10,0,-1):
    t = (activation_prime_norm**(L-l-1))*(W.abs().max()**(L-l-1))*W2.abs()
    maxS.append(t.max())
print(maxS)


[tensor(0.2822), tensor(1.1821), tensor(0.2822), tensor(0.0674), tensor(0.0161), tensor(0.0038), tensor(0.0009), tensor(0.0002), tensor(5.2249e-05), tensor(1.2474e-05), tensor(2.9779e-06)]


In [22]:
activation_pprime_norm = 0.7698003847290195
hess_bound = activation_pprime_norm*sum([a*a*b for a,b in zip(dzldx, maxS)])
hess_bound

tensor(1622.1239)